# Iris Data Science Pipeline

Notebook ini mendemonstrasikan seluruh alur Data Science menggunakan dataset Iris: analisis awal, validasi missing value, pembersihan data, transformasi fitur, pelabelan, visualisasi, dan penyimpanan hasil.

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

root = Path('.').resolve()
src_path = root / 'src'
sys.path.append(str(src_path))

from generate_synthetic_dataset import ensure_dataset

data_dir = root / 'data'
data_dir.mkdir(exist_ok=True)
source_path = data_dir / 'iris_data.csv'
report_dir = root / 'reports'
report_dir.mkdir(exist_ok=True)

df = ensure_dataset(source_path)
df.head()

## Langkah 1: Analisis dan Validasi Data

Analisis awal ini mencakup ringkasan dataset, missing value, duplikasi, dan outlier.

print('Jumlah baris:', len(df))
print('Jumlah kolom:', len(df.columns))
print('
Kolom dan tipe data:')
print(df.dtypes)

print('
Missing value per kolom:')
print(df.isna().sum())

print('
Duplikasi:')
print(df.duplicated().sum())

print('
Statistik numerik:')
print(df.describe())

def count_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((df[col] < lower) | (df[col] > upper)).sum()

for col in df.select_dtypes(include=[np.number]).columns:
    print(f'{col}:', count_outliers(df, col), 'outlier')

## Langkah 2 dan 3: Strategi Pembersihan dan Koreksi Data

Dataset Iris ini akan dibersihkan dengan: 
- menghapus duplikasi 
- menghapus baris dengan terlalu banyak missing value 
- mengisi missing numeric dengan median 
- mengisi missing kategorikal dengan mode 
- mengklipping nilai numerik ke kisaran realistis.

def clean_iris(df):
    df = df.copy().drop_duplicates().reset_index(drop=True)
    df = df[df.isna().sum(axis=1) <= 2].reset_index(drop=True)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include=['object', 'category']).columns:
        df[col] = df[col].astype('string')
        if not df[col].mode(dropna=True).empty:
            df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])
    ranges = {
        'sepal length (cm)': (3.0, 8.0),
        'sepal width (cm)': (2.0, 5.0),
        'petal length (cm)': (1.0, 7.0),
        'petal width (cm)': (0.1, 3.0),
    }
    for col, (low, high) in ranges.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=low, upper=high)
    return df

df_clean = clean_iris(df)
print('Jumlah baris setelah pembersihan:', len(df_clean))
print(df_clean.isna().sum())

## Langkah 4: Transformasi Data dan Rekayasa Fitur

Transformasi mencakup pembuatan luas sepal/petal, rasio area, normalisasi numerik, dan encoding label spesies.

def transform_iris(df):
    df = df.copy()
    df['sepal_area'] = (df['sepal length (cm)'] * df['sepal width (cm)']).round(3)
    df['petal_area'] = (df['petal length (cm)'] * df['petal width (cm)']).round(3)
    df['area_ratio'] = (df['petal_area'] / df['sepal_area']).round(3)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    df[[f'scaled_{col}' for col in numeric_cols]] = scaler.fit_transform(df[numeric_cols])
    df = pd.get_dummies(df, columns=['species'], drop_first=True)
    return df

df_transformed = transform_iris(df_clean)
df_transformed.head()

## Langkah 6 dan 7: Pelabelan Data dan Laporan Label

Pelabelan digunakan untuk membuat kategori `petal_size_category` dan `shape_label` berdasarkan fitur kalkulasi area.

def label_iris(df):
    df = df.copy()
    df['petal_size_category'] = pd.qcut(df['petal_area'], q=3, labels=['small', 'medium', 'large']).astype(str)
    df['shape_label'] = np.where(df['petal_area'] > df['sepal_area'], 'petal_dominant', 'sepal_dominant')
    return df

df_labeled = label_iris(df_transformed)
df_labeled[['petal_area', 'sepal_area', 'petal_size_category', 'shape_label']].head()

label_counts = df_labeled['petal_size_category'].value_counts().sort_index()
print(label_counts)

## Langkah 8: Visualisasi Data

Visualisasi membantu memahami distribusi fitur, outlier, dan perbedaan antar label.

sns.set_theme(style='whitegrid')
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['sepal length (cm)'], kde=True, color='#4c72b0')
plt.title('Distribusi Sepal Length')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='species', y='petal length (cm)', palette='Set2')
plt.title('Boxplot Petal Length by Species')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_labeled, x='sepal length (cm)', y='sepal width (cm)', hue='petal_size_category', palette='Set2')
plt.title('Sepal Length vs Sepal Width by Petal Size Category')
plt.show()

## Langkah 9: Evaluasi dan Dokumentasi

Menyimpan hasil pembersihan dan transformasi untuk digunakan dalam model atau laporan lebih lanjut.

cleaned_path = data_dir / 'iris_data_cleaned.csv'
transformed_path = data_dir / 'iris_data_transformed.csv'
report_path = report_dir / 'analysis_report.md'
distribution_path = report_dir / 'label_distribution.csv'

df_clean.to_csv(cleaned_path, index=False)
df_labeled.to_csv(transformed_path, index=False)
label_counts.to_frame('count').to_csv(distribution_path, index=True)

report_lines = [
    '# Laporan Analisis Iris',
    f'Jumlah baris awal: {len(df)}',
    f'Jumlah baris bersih: {len(df_clean)}',
    'Missing value telah diimputasi dengan median untuk fitur numerik.',
    'Fitur baru dibuat: sepal_area, petal_area, area_ratio.',
    'Label baru dibuat: petal_size_category dan shape_label.',
]
report_path.write_text('
'.join(report_lines), encoding='utf-8')
print('Saved files:')
print(cleaned_path)
print(transformed_path)
print(distribution_path)